Schema Movie

{

  "_id": "m_1",
  
  "title": "Toy Story",
  
  "year": 1995,
  
  "countries": ["USA"],
  
  "genres": ["Adventure", "Animation", "Children"],
  
  "topTags": [
  
    { "tag": "pixar", "relevance": 0.97 },
  
    { "tag": "animation", "relevance": 0.89 },
  
    { "tag": "disney", "relevance": 0.85 },
  
    { "tag": "toys", "relevance": 0.82 },
  
    { "tag": "bright", "relevance": 0.75 },
  
    { "tag": "witty", "relevance": 0.72 },
  
    { "tag": "cgi", "relevance": 0.68 },
  
    { "tag": "funny", "relevance": 0.65 },
  
    { "tag": "friendship", "relevance": 0.61 },
  
    { "tag": "classic", "relevance": 0.58 }
  
  ]
}

Schema Ratings

{
 
  "_id": ObjectId("65f1..."),
 
  "userId": 543,
 
  "movieId": "m_1",
 
  "rating": 4.5,
 
  "timestamp": 1147880044

}


Motivazione:
MongoDB impone un limite fisico di 16 Megabyte per ogni singolo documento, se provassimo a inserire i 25 milioni di rating all'interno del rispettivo documento film, i film più popolari supererebbero immediatamente i 16MB di dimensione, causando il blocco del database e rendendo impossibile l'aggiornamento dei dati. Quindi, separando i ratings in una collezione dedicata, rendiamo il sistema orizzontalmente scalabile. Non c'è limite al numero di rating che un film può ricevere.

Il dataset originale fornisce oltre 1.100 punteggi di rilevanza per ogni film. Inserire tutti i 1.100 tag renderebbe il documento del film pesante e rumoroso, contenendo molti tag con rilevanza quasi nulla (es. tag "horror" per un film Disney). Questo rallenterebbe il caricamento della RAM e il trasferimento dei dati in rete. Di conseguenza si selezionano solo i Top 10 tag per rilevanza. Questa scelta garantisce che il documento movies rimanga snello, fornendo però all'utente le informazioni descrittive più importanti e significative del film con un'unica operazione di lettura.

A differenza dei database SQL, dove "Paesi" e "Generi" richiederebbero tabelle separate e JOIN costose, qui abbiamo scelto la denormalizzazione controllata. Inserire i nomi reali dei generi e dei paesi direttamente nel documento film elimina la necessità di operazioni $lookup (le join di MongoDB) per le query di base.

ETL

In [1]:
import pandas as pd
import json
import re
import ast

def extract_country_names(val):
    try:
        if isinstance(val, str):
            data = ast.literal_eval(val)
        else:
            data = val
        return [c['name'] for c in data] if isinstance(data, list) else []
    except:
        return []

print("1/4 Caricamento file base...")
movies = pd.read_csv('movies.csv')
links = pd.read_csv('links.csv')
kaggle_tmdb = pd.read_csv('tmdb_movies_data.csv')
tags_dict = pd.read_csv('genome-tags.csv')

# --- ELABORAZIONE TAG (CHUNKING LOW-RAM) ---
print("2/4 Elaborazione Tag (circa 1-2 minuti)...")
chunk_size = 2_000_000 
top_tags_list = []

for i, chunk in enumerate(pd.read_csv('genome-scores.csv', chunksize=chunk_size)):
    # Unisce coi nomi dei tag
    chunk = chunk.merge(tags_dict, on='tagId')
    # Prende i Top 10 LOCALI per ogni film in questo chunk
    chunk_top = chunk.sort_values(['movieId', 'relevance'], ascending=[True, False]).groupby('movieId').head(10)
    top_tags_list.append(chunk_top)

# Unisce tutti i frammenti
combined_tags = pd.concat(top_tags_list)
# Estrae i Top 10 GLOBALI definitivi
final_top_tags = (combined_tags.sort_values(['movieId', 'relevance'], ascending=[True, False])
                  .groupby('movieId')
                  .head(10)
                  .groupby('movieId')
                  .apply(lambda x: x[['tag', 'relevance']].to_dict('records'))
                  .reset_index(name='topTags'))

# --- ARRICCHIMENTO E PULIZIA ---
print("3/4 Join con Kaggle per i Paesi...")
movies = movies.merge(links[['movieId', 'tmdbId']], on='movieId', how='left')

kaggle_key = 'id' if 'id' in kaggle_tmdb.columns else 'tmdbId'
movies = movies.merge(kaggle_tmdb[[kaggle_key, 'production_countries']], 
                      left_on='tmdbId', right_on=kaggle_key, how='left')

movies['countries'] = movies['production_countries'].apply(extract_country_names)

print("4/4 Pulizia Titoli e Generi...")
# Regex a prova di bomba: cerca l'ultimo blocco di 4 cifre tra parentesi
movies['year'] = movies['title'].str.extract(r'.*\((\d{4})\)')[0].astype('Int64') # Int64 gestisce i valori Null/NaN elegantemente
# Pulisce l'anno dal titolo
movies['title'] = movies['title'].apply(lambda x: re.sub(r'\s*\(\d{4}\)', '', x).strip())
# Filtra i generi fasulli
movies['genres'] = movies['genres'].apply(lambda x: x.split('|') if pd.notnull(x) and x != '(no genres listed)' else [])

# Unione finale col risultato dei Tag
movies_final = movies.merge(final_top_tags, on='movieId', how='left')
movies_final.rename(columns={'movieId': '_id'}, inplace=True)

# Se un film non ha tag, mettiamo un array vuoto invece di NaN
movies_final['topTags'] = movies_final['topTags'].apply(lambda x: x if isinstance(x, list) else [])

# --- SALVATAGGIO ---
print("Salvataggio JSON finale...")
cols = ['_id', 'title', 'year', 'countries', 'genres', 'topTags']
# force_ascii=False mantiene intatti gli accenti o caratteri speciali nei titoli stranieri
movies_final[cols].to_json('movies_clean.json', orient='records', force_ascii=False)
print("✅ File 'movies_clean.json' pronto al 100% per MongoDB!")

1/4 Caricamento file base...
2/4 Elaborazione Tag (circa 1-2 minuti)...
3/4 Join con Kaggle per i Paesi...
4/4 Pulizia Titoli e Generi...
Salvataggio JSON finale...
✅ File 'movies_clean.json' pronto al 100% per MongoDB!


Verrà utilizzato un container su Docker

#Copia il file dei film

docker cp movies_clean.json mongodb:/tmp/movies_clean.json

#Copia il file dei rating

docker cp ratings.csv mongodb:/tmp/ratings.csv

IMPORT COLLECTION Movies

docker exec mongodb mongoimport --db movielens --collection movies --file /tmp/movies_clean.json --jsonArray

IMPORT COLLECTION Ratings

docker exec mongodb mongoimport --db movielens --collection ratings --type csv --file /tmp/ratings.csv --headerline

CREAZIONE INDICI

// Indice per velocizzare le join (lookup) tra rating e film

db.ratings.createIndex({ movieId: 1 });

// Indice per velocizzare le analisi temporali (punto 1.2.3)

db.ratings.createIndex({ timestamp: 1 });

// Indice per i generi dei film (punti 1.2.2 e 1.2.3)

db.movies.createIndex({ genres: 1 });

Le query dovranno essere eseguite nella shell del container

QUERY 1.2.1

In [ ]:
db.movies.aggregate([
  // 1. Filtriamo solo i film prodotti tra il 1980 e il 2019 (le decadi richieste)
  { 
    $match: { 
      year: { $gte: 1980, $lt: 2020 } 
    } 
  },

  // 2. Poiché 'countries' è un array, usiamo $unwind. 
  // Se un film è una co-produzione (es. USA e Francia), verrà contato per entrambi i paesi.
  { $unwind: "$countries" },

  // 3. Calcoliamo la decade. 
  // Formula: anno - (anno modulo 10) -> es: 1995 - (5) = 1990.
  // Poi lo trasformiamo in stringa e aggiungiamo la "s" (es. "1990s").
  {
    $addFields: {
      decade: {
        $concat: [
          { $toString: { $subtract: ["$year", { $mod: ["$year", 10] }] } },
          "s"
        ]
      }
    }
  },

  // 4. Raggruppiamo per Paese e per Decade, contando i documenti
  {
    $group: {
      _id: {
        country: "$countries",
        decade: "$decade"
      },
      movieCount: { $sum: 1 }
    }
  },

  // 5. Ordiniamo per nome del Paese e poi cronologicamente per decade
  { 
    $sort: { 
      "_id.country": 1, 
      "_id.decade": 1 
    } 
  },

  // 6. Puliamo l'output finale per renderlo leggibile
  {
    $project: {
      _id: 0,
      country: "$_id.country",
      decade: "$_id.decade",
      movieCount: 1
    }
  }
])

OUTPUT

[
  { movieCount: 1, country: 'Afghanistan', decade: '2000s' },
  { movieCount: 1, country: 'Angola', decade: '2010s' },
  { movieCount: 1, country: 'Argentina', decade: '1990s' },
  { movieCount: 6, country: 'Argentina', decade: '2000s' },
  { movieCount: 2, country: 'Argentina', decade: '2010s' },
  { movieCount: 1, country: 'Aruba', decade: '1990s' },
  { movieCount: 6, country: 'Australia', decade: '1980s' },
  { movieCount: 19, country: 'Australia', decade: '1990s' },
  { movieCount: 53, country: 'Australia', decade: '2000s' },
  { movieCount: 28, country: 'Australia', decade: '2010s' },
  { movieCount: 1, country: 'Austria', decade: '1990s' },
  { movieCount: 4, country: 'Austria', decade: '2000s' },
  { movieCount: 1, country: 'Austria', decade: '2010s' },
  { movieCount: 1, country: 'Bahamas', decade: '2000s' },
  { movieCount: 1, country: 'Bahamas', decade: '2010s' },
  { movieCount: 1, country: 'Belgium', decade: '1990s' },
  { movieCount: 12, country: 'Belgium', decade: '2000s' },
  { movieCount: 10, country: 'Belgium', decade: '2010s' },
  { movieCount: 1, country: 'Bhutan', decade: '2000s' },
  { movieCount: 1, country: 'Bolivia', decade: '2010s' }
]
Type "it" for more
movielens> it
[
  { movieCount: 1, country: 'Bosnia and Herzegovina', decade: '2000s' },
  { movieCount: 2, country: 'Brazil', decade: '1990s' },
  { movieCount: 4, country: 'Brazil', decade: '2000s' },
  { movieCount: 6, country: 'Brazil', decade: '2010s' },
  { movieCount: 1, country: 'Bulgaria', decade: '2000s' },
  { movieCount: 3, country: 'Bulgaria', decade: '2010s' },
  { movieCount: 1, country: 'Cambodia', decade: '2010s' },
  { movieCount: 1, country: 'Cameroon', decade: '2000s' },
  { movieCount: 14, country: 'Canada', decade: '1980s' },
  { movieCount: 30, country: 'Canada', decade: '1990s' },
  { movieCount: 124, country: 'Canada', decade: '2000s' },
  { movieCount: 82, country: 'Canada', decade: '2010s' },
  { movieCount: 2, country: 'Chile', decade: '2010s' },
  { movieCount: 1, country: 'China', decade: '1980s' },
  { movieCount: 1, country: 'China', decade: '1990s' },
  { movieCount: 24, country: 'China', decade: '2000s' },
  { movieCount: 31, country: 'China', decade: '2010s' },
  { movieCount: 1, country: 'Colombia', decade: '2000s' },
  { movieCount: 1, country: 'Colombia', decade: '2010s' },
  { movieCount: 1, country: 'Cyprus', decade: '2010s' }
]
Type "it" for more
movielens> it
[
  { movieCount: 2, country: 'Czech Republic', decade: '1990s' },
  { movieCount: 17, country: 'Czech Republic', decade: '2000s' },
  { movieCount: 4, country: 'Czech Republic', decade: '2010s' },
  { movieCount: 3, country: 'Denmark', decade: '1990s' },
  { movieCount: 7, country: 'Denmark', decade: '2000s' },
  { movieCount: 9, country: 'Denmark', decade: '2010s' },
  { movieCount: 1, country: 'Dominica', decade: '2000s' },
  { movieCount: 1, country: 'Ecuador', decade: '2000s' },
  { movieCount: 1, country: 'Egypt', decade: '2010s' },
  { movieCount: 1, country: 'Fiji', decade: '1990s' },
  { movieCount: 3, country: 'Finland', decade: '2000s' },
  { movieCount: 2, country: 'Finland', decade: '2010s' },
  { movieCount: 4, country: 'France', decade: '1980s' },
  { movieCount: 31, country: 'France', decade: '1990s' },
  { movieCount: 158, country: 'France', decade: '2000s' },
  { movieCount: 101, country: 'France', decade: '2010s' },
  { movieCount: 4, country: 'Germany', decade: '1980s' },
  { movieCount: 25, country: 'Germany', decade: '1990s' },
  { movieCount: 241, country: 'Germany', decade: '2000s' },
  { movieCount: 42, country: 'Germany', decade: '2010s' }
]
Type "it" for more
movielens> it
[
  { movieCount: 2, country: 'Greece', decade: '2000s' },
  { movieCount: 1, country: 'Greece', decade: '2010s' },
  { movieCount: 1, country: 'Guadaloupe', decade: '2010s' },
  { movieCount: 2, country: 'Hong Kong', decade: '1980s' },
  { movieCount: 6, country: 'Hong Kong', decade: '1990s' },
  { movieCount: 21, country: 'Hong Kong', decade: '2000s' },
  { movieCount: 16, country: 'Hong Kong', decade: '2010s' },
  { movieCount: 2, country: 'Hungary', decade: '1990s' },
  { movieCount: 6, country: 'Hungary', decade: '2000s' },
  { movieCount: 4, country: 'Hungary', decade: '2010s' },
  { movieCount: 1, country: 'Iceland', decade: '1980s' },
  { movieCount: 3, country: 'Iceland', decade: '2000s' },
  { movieCount: 2, country: 'Iceland', decade: '2010s' },
  { movieCount: 1, country: 'India', decade: '1980s' },
  { movieCount: 3, country: 'India', decade: '1990s' },
  { movieCount: 17, country: 'India', decade: '2000s' },
  { movieCount: 22, country: 'India', decade: '2010s' },
  { movieCount: 1, country: 'Indonesia', decade: '2010s' },
  { movieCount: 1, country: 'Iran', decade: '1990s' },
  { movieCount: 1, country: 'Iran', decade: '2000s' }
]

QUERY 1.2.2

In [ ]:
db.ratings.aggregate([
  // 1. Join con la collection dei film
  { $lookup: { from: "movies", localField: "movieId", foreignField: "_id", as: "movie" } },
  { $unwind: "$movie" },
  
  // 2. Esplodiamo l'array dei generi (un rating vale per ogni genere del film)
  { $unwind: "$movie.genres" },

  // 3. Il Facet esegue due calcoli separati in contemporanea
  { $facet: {
      // Sub-pipeline A: Calcola Utenti Unici per Genere
      "utenti_unici": [
          // Raggruppa per Genere e Utente (elimina i duplicati di voto dello stesso utente nello stesso genere)
          { $group: { _id: { genre: "$movie.genres", userId: "$userId" } } },
          // Conta quanti utenti sono rimasti per ogni genere
          { $group: { _id: "$_id.genre", uniqueUsers: { $sum: 1 } } }
      ],
      // Sub-pipeline B: Calcola Totale Rating e Classifica dei Titoli
      "statistiche_titoli": [
          // Conta i voti per ogni singolo titolo in ogni genere
          { $group: { _id: { genre: "$movie.genres", title: "$movie.title" }, count: { $sum: 1 } } },
          // Ordina i titoli dal più votato al meno votato
          { $sort: { count: -1 } },
          // Raggruppa per Genere: Somma tutti i voti e accumula la classifica
          { $group: {
              _id: "$_id.genre",
              totalRatings: { $sum: "$count" },
              topTitles: { $push: { title: "$_id.title", count: "$count" } }
          }}
      ]
  }},
  
  // 4. Formattazione dell'output (Uniamo i risultati dei due Facet)
  { $unwind: "$utenti_unici" },
  { $unwind: "$statistiche_titoli" },
  // Assicuriamoci di unire le righe che corrispondono allo stesso genere
  { $match: { $expr: { $eq: ["$utenti_unici._id", "$statistiche_titoli._id"] } } },
  
  // 5. Proiezione Finale: Tagliamo l'array dei titoli tenendo solo i Top 5
  { $project: {
      _id: 0,
      genre: "$utenti_unici._id",
      uniqueUsers: "$utenti_unici.uniqueUsers",
      totalRatings: "$statistiche_titoli.totalRatings",
      top5Titles: { $slice: ["$statistiche_titoli.topTitles", 5] }
  }},
  { $sort: { totalRatings: -1 } }

], { allowDiskUse: true })

OUTPUT

[   
  {
    genre: 'Drama',
    uniqueUsers: 162519,
    totalRatings: 10962833,
    top5Titles: [
      { title: 'Forrest Gump', count: 81491 },
      { title: 'Shawshank Redemption, The', count: 81482 },
      { title: 'Pulp Fiction', count: 79672 },
      { title: "Schindler's List", count: 60411 },
      { title: 'Braveheart', count: 59184 }
    ]
  },
  {
    genre: 'Comedy',
    uniqueUsers: 162381,
    totalRatings: 8926230,
    top5Titles: [
      { title: 'Forrest Gump', count: 81491 },
      { title: 'Pulp Fiction', count: 79672 },
      { title: 'Toy Story', count: 57309 },
      { title: 'Back to the Future', count: 49595 },
      { title: 'Fargo', count: 47823 }
    ]
  },
  {
    genre: 'Action',
    uniqueUsers: 161975,
    totalRatings: 7446918,
    top5Titles: [
      { title: 'Matrix, The', count: 72674 },
      { title: 'Star Wars: Episode IV - A New Hope', count: 68717 },
      { title: 'Jurassic Park', count: 64144 },
      { title: 'Braveheart', count: 59184 },
      { title: 'Fight Club', count: 58773 }
    ]
  },
  {
    genre: 'Thriller',
    uniqueUsers: 161948,
    totalRatings: 6763272,
    top5Titles: [
      { title: 'Pulp Fiction', count: 79672 },
      { title: 'Silence of the Lambs, The', count: 74127 },
      { title: 'Matrix, The', count: 72674 },
      { title: 'Jurassic Park', count: 64144 },
      { title: 'Fight Club', count: 58773 }
    ]
  },
  {
    genre: 'Adventure',
    uniqueUsers: 161821,
    totalRatings: 5832424,
    top5Titles: [
      { title: 'Star Wars: Episode IV - A New Hope', count: 68717 },
      { title: 'Jurassic Park', count: 64144 },
      {
        title: 'Star Wars: Episode V - The Empire Strikes Back',
        count: 57361
      },
      { title: 'Toy Story', count: 57309 },
      {
        title: 'Lord of the Rings: The Fellowship of the Ring, The',
        count: 55736
      }
    ]
  },
  {
    genre: 'Romance',
    uniqueUsers: 161068,
    totalRatings: 4497291,
    top5Titles: [
      { title: 'Forrest Gump', count: 81491 },
      { title: 'American Beauty', count: 53689 },
      { title: 'Shrek', count: 42303 },
      { title: 'True Lies', count: 41673 },
      { title: 'Speed', count: 41302 }
    ]
  },
  {
    genre: 'Sci-Fi',
    uniqueUsers: 160063,
    totalRatings: 4325740,
    top5Titles: [
      { title: 'Matrix, The', count: 72674 },
      { title: 'Star Wars: Episode IV - A New Hope', count: 68717 },
      { title: 'Jurassic Park', count: 64144 },
      { title: 'Terminator 2: Judgment Day', count: 57379 },
      {
        title: 'Star Wars: Episode V - The Empire Strikes Back',
        count: 57361
      }
    ]
  },
  {
    genre: 'Crime',
    uniqueUsers: 160855,
    totalRatings: 4190259,
    top5Titles: [
      { title: 'Shawshank Redemption, The', count: 81482 },
      { title: 'Pulp Fiction', count: 79672 },
      { title: 'Silence of the Lambs, The', count: 74127 },
      { title: 'Fight Club', count: 58773 },
      { title: 'Usual Suspects, The', count: 55366 }
    ]
  },
  {
    genre: 'Fantasy',
    uniqueUsers: 156809,
    totalRatings: 2831585,
    top5Titles: [
      { title: 'Toy Story', count: 57309 },
      {
        title: 'Lord of the Rings: The Fellowship of the Ring, The',
        count: 55736
      },
      { title: 'Lord of the Rings: The Two Towers, The', count: 51138 },
      {
        title: 'Lord of the Rings: The Return of the King, The',
        count: 50797
      },
      { title: 'Shrek', count: 42303 }
    ]
  },
  {
    genre: 'Children',
    uniqueUsers: 148391,
    totalRatings: 2124258,
    top5Titles: [
      { title: 'Toy Story', count: 57309 },
      { title: 'Aladdin', count: 43387 },
      { title: 'Lion King, The', count: 42745 },
      { title: 'Shrek', count: 42303 },
      { title: 'Beauty and the Beast', count: 35736 }
    ]
  },
  {
    genre: 'Mystery',
    uniqueUsers: 152209,
    totalRatings: 2010995,
    top5Titles: [
      { title: 'Usual Suspects, The', count: 55366 },
      { title: 'Seven (a.k.a. Se7en)', count: 50596 },
      { title: 'Twelve Monkeys (a.k.a. 12 Monkeys)', count: 47054 },
      { title: 'Sixth Sense, The', count: 46713 },
      { title: 'Memento', count: 41195 }
    ]
  },
  {
    genre: 'Horror',
    uniqueUsers: 141445,
    totalRatings: 1892183,
    top5Titles: [
      { title: 'Silence of the Lambs, The', count: 74127 },
      { title: 'Sixth Sense, The', count: 46713 },
      { title: 'Alien', count: 36357 },
      { title: 'Aliens', count: 31684 },
      { title: 'Shining, The', count: 29931 }
    ]
  },
  {
    genre: 'Animation',
    uniqueUsers: 140225,
    totalRatings: 1630987,
    top5Titles: [
      { title: 'Toy Story', count: 57309 },
      { title: 'Aladdin', count: 43387 },
      { title: 'Lion King, The', count: 42745 },
      { title: 'Shrek', count: 42303 },
      { title: 'Beauty and the Beast', count: 35724 }
    ]
  },
  {
    genre: 'War',
    uniqueUsers: 146905,
    totalRatings: 1267346,
    top5Titles: [
      { title: 'Forrest Gump', count: 81491 },
      { title: "Schindler's List", count: 60411 },
      { title: 'Braveheart', count: 59184 },
      { title: 'Saving Private Ryan', count: 46783 },
      {
        title: 'Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb',
        count: 26714
      }
    ]
  },
  {
    genre: 'IMAX',
    uniqueUsers: 119012,
    totalRatings: 1063279,
    top5Titles: [
      { title: 'Apollo 13', count: 48377 },
      { title: 'Lion King, The', count: 42745 },
      { title: 'Dark Knight, The', count: 41519 },
      { title: 'Inception', count: 38895 },
      { title: 'Beauty and the Beast', count: 35723 }
    ]
  },
  {
    genre: 'Musical',
    uniqueUsers: 126133,
    totalRatings: 964252,
    top5Titles: [
      { title: 'Aladdin', count: 43373 },
      { title: 'Lion King, The', count: 42745 },
      { title: 'Beauty and the Beast', count: 35723 },
      { title: 'Willy Wonka & the Chocolate Factory', count: 28079 },
      { title: 'Wizard of Oz, The', count: 24658 }
    ]
  },
  {
    genre: 'Western',
    uniqueUsers: 110037,
    totalRatings: 483731,
    top5Titles: [
      { title: 'Dances with Wolves', count: 41615 },
      { title: 'Back to the Future Part III', count: 21881 },
      { title: 'Django Unchained', count: 20687 },
      {
        title: 'Good, the Bad and the Ugly, The (Buono, il brutto, il cattivo, Il)',
        count: 18162
      },
      { title: 'Maverick', count: 16475 }
    ]
  },
  {
    genre: 'Documentary',
    uniqueUsers: 60393,
    totalRatings: 322449,
    top5Titles: [
      { title: 'Bowling for Columbine', count: 13619 },
      { title: 'Super Size Me', count: 10913 },
      { title: 'Hoop Dreams', count: 9739 },
      { title: 'Fahrenheit 9/11', count: 9253 },
      { title: 'Roger & Me', count: 6898 }
    ]
  },
  {
    genre: 'Film-Noir',
    uniqueUsers: 67849,
    totalRatings: 247227,
    top5Titles: [
      { title: 'L.A. Confidential', count: 27749 },
      { title: 'Sin City', count: 22271 },
      { title: 'Chinatown', count: 16929 },
      { title: 'Mulholland Drive', count: 12451 },
      { title: 'Maltese Falcon, The', count: 12257 }
    ]
  }
]

QUERY 1.2.3

In [ ]:
db.ratings.aggregate([
  // 1. Join con i film per ottenere i generi
  { $lookup: { from: "movies", localField: "movieId", foreignField: "_id", as: "movie" } },
  { $unwind: "$movie" },
  { $unwind: "$movie.genres" },

  // 2. Trasformazione del timestamp da intero a Oggetto Data (ISODate)
  { $addFields: {
      actualDate: { $toDate: { $multiply: ["$timestamp", 1000] } }
  }},

  // 3. Raggruppamento per Genere, Anno e Mese (estratti dinamicamente)
  { $group: {
      _id: {
          genre: "$movie.genres",
          year: { $year: "$actualDate" },
          month: { $month: "$actualDate" }
      },
      volume: { $sum: 1 },
      avgRating: { $avg: "$rating" }
  }},

  // 4. Pulizia dei dati in uscita e arrotondamento della media a 2 decimali
  { $project: {
      _id: 0,
      genre: "$_id.genre",
      year: "$_id.year",
      month: "$_id.month",
      volume: 1,
      avgRating: { $round: ["$avgRating", 2] }
  }},
  
  // 5. Ordinamento logico: per Genere, poi cronologico (dal più vecchio al più recente)
  { $sort: { genre: 1, year: 1, month: 1 } }

], { allowDiskUse: true })

OUTPUT

[
  { volume: 5, genre: 'Action', year: 1996, month: 1, avgRating: 3.8 },
  { volume: 76, genre: 'Action', year: 1996, month: 2, avgRating: 3.3 },
  {
    volume: 2168,
    genre: 'Action',
    year: 1996,
    month: 3,
    avgRating: 3.74
  },
  {
    volume: 13576,
    genre: 'Action',
    year: 1996,
    month: 4,
    avgRating: 3.66
  },
  {
    volume: 50221,
    genre: 'Action',
    year: 1996,
    month: 5,
    avgRating: 3.36
  },
  {
    volume: 69878,
    genre: 'Action',
    year: 1996,
    month: 6,
    avgRating: 3.4
  },
  {
    volume: 62380,
    genre: 'Action',
    year: 1996,
    month: 7,
    avgRating: 3.44
  },
  {
    volume: 60453,
    genre: 'Action',
    year: 1996,
    month: 8,
    avgRating: 3.39
  },
  {
    volume: 41464,
    genre: 'Action',
    year: 1996,
    month: 9,
    avgRating: 3.39
  },
  {
    volume: 50775,
    genre: 'Action',
    year: 1996,
    month: 10,
    avgRating: 3.46
  },
  {
    volume: 50910,
    genre: 'Action',
    year: 1996,
    month: 11,
    avgRating: 3.47
  },
  {
    volume: 33397,
    genre: 'Action',
    year: 1996,
    month: 12,
    avgRating: 3.56
  },
  {
    volume: 26995,
    genre: 'Action',
    year: 1997,
    month: 1,
    avgRating: 3.52
  },
  {
    volume: 16143,
    genre: 'Action',
    year: 1997,
    month: 2,
    avgRating: 3.56
  },
  {
    volume: 24822,
    genre: 'Action',
    year: 1997,
    month: 3,
    avgRating: 3.56
  },
  {
    volume: 21294,
    genre: 'Action',
    year: 1997,
    month: 4,
    avgRating: 3.57
  },
  {
    volume: 25178,
    genre: 'Action',
    year: 1997,
    month: 5,
    avgRating: 3.58
  },
  {
    volume: 21548,
    genre: 'Action',
    year: 1997,
    month: 6,
    avgRating: 3.53
  },
  {
    volume: 15444,
    genre: 'Action',
    year: 1997,
    month: 7,
    avgRating: 3.55
  },
  {
    volume: 4603,
    genre: 'Action',
    year: 1997,
    month: 9,
    avgRating: 3.49
  }
]
Type "it" for more
movielens> it
[
  {
    volume: 5275,
    genre: 'Action',
    year: 1997,
    month: 10,
    avgRating: 3.51
  },
  {
    volume: 8749,
    genre: 'Action',
    year: 1997,
    month: 11,
    avgRating: 3.41
  },
  {
    volume: 4604,
    genre: 'Action',
    year: 1997,
    month: 12,
    avgRating: 3.4
  },
  {
    volume: 5584,
    genre: 'Action',
    year: 1998,
    month: 1,
    avgRating: 3.38
  },
  {
    volume: 3633,
    genre: 'Action',
    year: 1998,
    month: 2,
    avgRating: 3.37
  },
  {
    volume: 4098,
    genre: 'Action',
    year: 1998,
    month: 3,
    avgRating: 3.39
  },
  {
    volume: 4075,
    genre: 'Action',
    year: 1998,
    month: 4,
    avgRating: 3.43
  },
  {
    volume: 2201,
    genre: 'Action',
    year: 1998,
    month: 5,
    avgRating: 3.31
  },
  {
    volume: 3184,
    genre: 'Action',
    year: 1998,
    month: 6,
    avgRating: 3.31
  },
  {
    volume: 19847,
    genre: 'Action',
    year: 1998,
    month: 7,
    avgRating: 3.51
  },
  {
    volume: 8980,
    genre: 'Action',
    year: 1998,
    month: 8,
    avgRating: 3.47
  },
  {
    volume: 4873,
    genre: 'Action',
    year: 1998,
    month: 9,
    avgRating: 3.39
  },
  {
    volume: 3696,
    genre: 'Action',
    year: 1998,
    month: 10,
    avgRating: 3.44
  },
  {
    volume: 3754,
    genre: 'Action',
    year: 1998,
    month: 11,
    avgRating: 3.55
  },
  {
    volume: 7876,
    genre: 'Action',
    year: 1998,
    month: 12,
    avgRating: 3.39
  },
  {
    volume: 6231,
    genre: 'Action',
    year: 1999,
    month: 1,
    avgRating: 3.35
  },
  {
    volume: 4260,
    genre: 'Action',
    year: 1999,
    month: 2,
    avgRating: 3.52
  },
  {
    volume: 2250,
    genre: 'Action',
    year: 1999,
    month: 3,
    avgRating: 3.44
  },
  {
    volume: 3575,
    genre: 'Action',
    year: 1999,
    month: 4,
    avgRating: 3.52
  },
  {
    volume: 3800,
    genre: 'Action',
    year: 1999,
    month: 5,
    avgRating: 3.43
  }
]
Type "it" for more
movielens> it
[
  {
    volume: 3006,
    genre: 'Action',
    year: 1999,
    month: 6,
    avgRating: 3.46
  },
  { volume: 2350, genre: 'Action', year: 1999, month: 7, avgRating: 3.5 },
  {
    volume: 3055,
    genre: 'Action',
    year: 1999,
    month: 8,
    avgRating: 3.32
  },
  {
    volume: 6375,
    genre: 'Action',
    year: 1999,
    month: 9,
    avgRating: 3.41
  },
  {
    volume: 60812,
    genre: 'Action',
    year: 1999,
    month: 10,
    avgRating: 3.38
  },
  {
    volume: 48055,
    genre: 'Action',
    year: 1999,
    month: 11,
    avgRating: 3.43
  },
  {
    volume: 107659,
    genre: 'Action',
    year: 1999,
    month: 12,
    avgRating: 3.51
  },
  {
    volume: 36542,
    genre: 'Action',
    year: 2000,
    month: 1,
    avgRating: 3.47
  },
  {
    volume: 26743,
    genre: 'Action',
    year: 2000,
    month: 2,
    avgRating: 3.51
  },
  {
    volume: 32304,
    genre: 'Action',
    year: 2000,
    month: 3,
    avgRating: 3.48
  },
  {
    volume: 37318,
    genre: 'Action',
    year: 2000,
    month: 4,
    avgRating: 3.53
  },
  {
    volume: 24791,
    genre: 'Action',
    year: 2000,
    month: 5,
    avgRating: 3.49
  },
  {
    volume: 20193,
    genre: 'Action',
    year: 2000,
    month: 6,
    avgRating: 3.47
  },
  {
    volume: 33062,
    genre: 'Action',
    year: 2000,
    month: 7,
    avgRating: 3.47
  },
  {
    volume: 60534,
    genre: 'Action',
    year: 2000,
    month: 8,
    avgRating: 3.49
  },
  {
    volume: 17804,
    genre: 'Action',
    year: 2000,
    month: 9,
    avgRating: 3.46
  },
  {
    volume: 13835,
    genre: 'Action',
    year: 2000,
    month: 10,
    avgRating: 3.44
  },
  {
    volume: 91995,
    genre: 'Action',
    year: 2000,
    month: 11,
    avgRating: 3.44
  },
  {
    volume: 37598,
    genre: 'Action',
    year: 2000,
    month: 12,
    avgRating: 3.45
  },
  {
    volume: 29429,
    genre: 'Action',
    year: 2001,
    month: 1,
    avgRating: 3.45
  }
]
Type "it" for more
movielens> it
[
  {
    volume: 17813,
    genre: 'Action',
    year: 2001,
    month: 2,
    avgRating: 3.51
  },
  {
    volume: 16176,
    genre: 'Action',
    year: 2001,
    month: 3,
    avgRating: 3.4
  },
  {
    volume: 12791,
    genre: 'Action',
    year: 2001,
    month: 4,
    avgRating: 3.47
  },
  {
    volume: 11673,
    genre: 'Action',
    year: 2001,
    month: 5,
    avgRating: 3.41
  },
  {
    volume: 41348,
    genre: 'Action',
    year: 2001,
    month: 6,
    avgRating: 3.51
  },
  {
    volume: 35921,
    genre: 'Action',
    year: 2001,
    month: 7,
    avgRating: 3.45
  },
  {
    volume: 25111,
    genre: 'Action',
    year: 2001,
    month: 8,
    avgRating: 3.45
  },
  {
    volume: 22270,
    genre: 'Action',
    year: 2001,
    month: 9,
    avgRating: 3.38
  },
  {
    volume: 18127,
    genre: 'Action',
    year: 2001,
    month: 10,
    avgRating: 3.45
  },
  {
    volume: 19156,
    genre: 'Action',
    year: 2001,
    month: 11,
    avgRating: 3.48
  },
  {
    volume: 20551,
    genre: 'Action',
    year: 2001,
    month: 12,
    avgRating: 3.49
  },